# Entendimento do negócio

In [84]:
# Importações
import ee
import requests
import re
import os
import urllib3
import zipfile
import shutil

import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np

from glob import glob
from io import BytesIO
from shapely.geometry import box

print("OK")

OK


In [50]:
# Roda o comando no terminal : earthengine authenticate --auth_mode=notebook
ee.Authenticate()
ee.Initialize(project="spatial-yew-490017-r3")
print(ee.String("Hello from the Earth Engine servers!").getInfo())

Hello from the Earth Engine servers!


# Entendimento dos dados

In [99]:
FILES_DIR = "files"

os.makedirs(FILES_DIR, exist_ok=True)

In [77]:
# Download UCs boundaries via WFS (INDE/ICMBio)

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

ucs_dir = os.path.join(FILES_DIR, "limites_ucs")
os.makedirs(ucs_dir, exist_ok=True)

typename_uc = "ICMBio:limiteucsfederais_a"
output_path = os.path.join(ucs_dir, "limites_ucs.geojson")

if os.path.exists(output_path):
    print(f"Aruivo ja existente: {output_path}")
else:
    wfs_url = (
        "https://geoservicos.inde.gov.br/geoserver/ICMBio/ows"
        f"?service=WFS&version=2.0.0&request=GetFeature"
        f"&typeName={typename_uc}&outputFormat=application/json"
    )

    response = requests.get(wfs_url, timeout=180, verify=False)
    response.raise_for_status()

    with open(output_path, 'wb') as f:
        f.write(response.content)

    print(f"Baixado em: {output_path}")

gdf_ucs = gpd.read_file(output_path)

print("\nShape:", gdf_ucs.shape)
print("\nColumns:", gdf_ucs.columns.tolist())
print("\nOriginal CRS:", gdf_ucs.crs)
print("\nFirst rows:\n", gdf_ucs.head())

Baixado em: files\limites_ucs\limites_ucs.geojson

Shape: (347, 22)

Columns: ['ogc_fid', 'id', 'nomeuc', 'cnuc', 'criacaoano', 'areahaalb', 'perimm', 'criacaoato', 'esferaadm', 'grupouc', 'biomas', 'gregional', 'fusoabrang', 'demarcacao', 'escalauc', 'bioma_pred', 'cat_iucn', 'uf', 'categoria_', 'sigla_cate', 'dominio', 'geometry']

Original CRS: EPSG:4674

First rows:
    ogc_fid    id                                             nomeuc  \
0        1  2085  RESERVA DE DESENVOLVIMENTO SUSTENTÁVEL CÓRREGO...   
1        2  2086         PARQUE NACIONAL DO PANTANAL MATO-GROSSENSE   
2        3  2088  ÁREA DE PROTEÇÃO AMBIENTAL MANANCIAIS DO RIO P...   
3        4  2089                        ESTAÇÃO ECOLÓGICA DE TAIAMÃ   
4        5  2091             PARQUE NACIONAL DA SERRA DAS CONFUSÕES   

           cnuc criacaoano      areahaalb        perimm  \
0  0000.00.5332       2026   40834.549015  1.878476e+05   
1  0000.00.0175       1981  183094.585492  2.797924e+05   
2  0000.00.1521       

In [ ]:
YEARS = [2025]  # example: [2023, 2025] would download 2023, 2024, and 2025

if len(YEARS) == 1:
    years_to_download = YEARS
else:
    years_to_download = list(range(min(YEARS), max(YEARS) + 1))

print("Years queued for download:", years_to_download)


Years queued for download: [2025]


In [97]:
# INPE FOCO DE CALOR - downlaod de range de ano

downloaded_files = {}
inpe_dir = os.path.join(FILES_DIR, "inpe_focos")
os.makedirs(inpe_dir, exist_ok=True)

for year in years_to_download:

    csv_filename = f"focos_br_todos-sats_{year}.csv"
    csv_path = os.path.join(inpe_dir, csv_filename)

    if os.path.exists(csv_path):
        print(f"[{year}] Already downloaded")
        downloaded_files[year] = csv_path
        continue

    zip_url = f"https://dataserver-coids.inpe.br/queimadas/queimadas/focos/csv/anual/Brasil_todos_sats/focos_br_todos-sats_{year}.zip"

    print(f"[{year}] Downloading")

    try:
        response = requests.get(zip_url, timeout=180)
        response.raise_for_status()

        tmp_dir = os.path.join(inpe_dir, f"tmp_{year}")
        os.makedirs(tmp_dir, exist_ok=True)

        with zipfile.ZipFile(BytesIO(response.content)) as z:
            z.extractall(tmp_dir)

        csv_files_found = glob(os.path.join(tmp_dir, "**", "*.csv"), recursive=True)

        if csv_files_found:
            shutil.move(csv_files_found[0], csv_path)
            downloaded_files[year] = csv_path
            print(f"[{year}] Saved")
        else:
            print(f"[{year}] No CSV found in zip")

        shutil.rmtree(tmp_dir, ignore_errors=True)

    except requests.exceptions.RequestException:
        print(f"[{year}] Download failed")

print("Done:", list(downloaded_files.keys()))

[2025] Downloading
[2025] Saved
Done: [2025]


In [ ]:
df_list = []
 
for year, path in downloaded_files.items():
    df_year = pd.read_csv(path)
    df_year['ano_arquivo'] = year
    df_list.append(df_year)

df_inpe = pd.concat(df_list, ignore_index=True)

print("Shape:", df_inpe.shape)
print("\nColumns:", df_inpe.columns.tolist())
print("\nData types:\n", df_inpe.dtypes)
print("\nFirst rows:\n", df_inpe.head())
print("\nYears present in data:", sorted(df_inpe['ano_arquivo'].unique()))


Shape: (3466399, 14)

Columns: ['latitude', 'longitude', 'data_pas', 'satelite', 'pais', 'estado', 'municipio', 'bioma', 'numero_dias_sem_chuva', 'precipitacao', 'risco_fogo', 'id_area_industrial', 'frp', 'ano_arquivo']

Data types:
 latitude                 float64
longitude                float64
data_pas                     str
satelite                     str
pais                         str
estado                       str
municipio                    str
bioma                        str
numero_dias_sem_chuva    float64
precipitacao             float64
risco_fogo               float64
id_area_industrial         int64
frp                      float64
ano_arquivo                int64
dtype: object

First rows:
    latitude  longitude             data_pas satelite    pais  \
0  -19.7334 -55.399899  2025-01-01 00:35:00  METOP-B  Brasil   
1  -15.2145 -60.173901  2025-01-01 00:38:00  METOP-B  Brasil   
2  -15.2090 -60.198799  2025-01-01 00:38:00  METOP-B  Brasil   
3  -12.2099 -49.3261

In [ ]:
# AAF ICMBio (Shapefile)

gdf_aaf = gpd.read_file('files/aaf_2025/aaf_2025.shp')

print("Shape:", gdf_aaf.shape)
print("\nColunas:", gdf_aaf.columns.tolist())
print("\nTipos de dado:\n", gdf_aaf.dtypes)


Shape: (3133, 25)

Colunas: ['cnuc', 'nome_uc', 'area_ha', 'acao', 'tipo', 'satelite', 'obs', 'juliano', 'data', 'ano', 'mes_nome', 'mes_num', 'categoria', 'local', 'area_uc', 'area_ent', 'bioma', 'gr_nome', 'ngi', 'cr', 'GlobalID', 'Shape__Are', 'Shape__Len', 'ct', 'geometry']

Tipos de dado:
 cnuc                     str
nome_uc                  str
area_ha              float64
acao                     str
tipo                     str
satelite                 str
obs                      str
juliano              float64
data          datetime64[ms]
ano                    int64
mes_nome                 str
mes_num                  str
categoria                str
local                    str
area_uc              float64
area_ent             float64
bioma                    str
gr_nome                  str
ngi                      str
cr                       str
GlobalID              object
Shape__Are           float64
Shape__Len           float64
ct                       str
geometry

# Preparação dos dados

In [105]:
METRIC_CRS = "EPSG:5880"  # SIRGAS 2000 / Brazil Polyconic
CELL_SIZE_M = 5000        # 5km x 5km, ajustável

In [106]:
#  Reprojetar UCs e AAF para CRS métrico

gdf_ucs_metric = gdf_ucs.to_crs(METRIC_CRS)
gdf_aaf_metric = gdf_aaf.to_crs(METRIC_CRS)

print("CRS UCs após reprojeção:", gdf_ucs_metric.crs)
print("CRS AAF após reprojeção:", gdf_aaf_metric.crs)


CRS UCs após reprojeção: EPSG:5880
CRS AAF após reprojeção: EPSG:5880


In [107]:
# Validar geometrias antes de seguir

gdf_ucs_metric['geometry'] = gdf_ucs_metric.geometry.buffer(0)
gdf_aaf_metric['geometry'] = gdf_aaf_metric.geometry.buffer(0)

print("\nApós correção com buffer(0):")
print("Geometrias inválidas em gdf_ucs_metric:", (~gdf_ucs_metric.is_valid).sum())
print("Geometrias inválidas em gdf_aaf_metric:", (~gdf_aaf_metric.is_valid).sum())



Após correção com buffer(0):
Geometrias inválidas em gdf_ucs_metric: 0
Geometrias inválidas em gdf_aaf_metric: 0


In [109]:
# Função para gerar grade de células dentro de uma UC

def generate_uc_grid(uc_geom, uc_id, cell_size=CELL_SIZE_M):
    minx, miny, maxx, maxy = uc_geom.bounds

    x_coords = np.arange(minx, maxx, cell_size)
    y_coords = np.arange(miny, maxy, cell_size)

    cells = []
    for i, x in enumerate(x_coords):
        for j, y in enumerate(y_coords):
            cell = box(x, y, x + cell_size, y + cell_size)
            if uc_geom.intersects(cell):
                clipped_cell = uc_geom.intersection(cell)
                if not clipped_cell.is_empty:
                    cells.append({
                        'cell_id': f"{uc_id}_{i}_{j}",
                        'uc_id': uc_id,
                        'geometry': clipped_cell
                    })
    return cells

In [110]:
# Aplicar a função para todas as UCs, construir grade completa

all_cells = []

for idx, row in gdf_ucs_metric.iterrows():
    uc_id = row['cnuc']
    uc_cells = generate_uc_grid(row.geometry, uc_id)
    all_cells.extend(uc_cells)

gdf_grid = gpd.GeoDataFrame(all_cells, crs=METRIC_CRS)

print("Total de células geradas:", len(gdf_grid))
print("\nExemplo de células:\n", gdf_grid.head())
print("\nCélulas por UC (top 10):\n", gdf_grid['uc_id'].value_counts().head(10))

Total de células geradas: 84492

Exemplo de células:
             cell_id         uc_id  \
0  0000.00.5332_0_1  0000.00.5332   
1  0000.00.5332_0_2  0000.00.5332   
2  0000.00.5332_0_3  0000.00.5332   
3  0000.00.5332_0_4  0000.00.5332   
4  0000.00.5332_1_0  0000.00.5332   

                                            geometry  
0  POLYGON ((6168555.065 8189810.869, 6168292.842...  
1  POLYGON ((6165673.44 8190005.065, 6165666.683 ...  
2  MULTIPOLYGON (((6164563.624 8195012.458, 61645...  
3  MULTIPOLYGON (((6163608.136 8199973.178, 61636...  
4  POLYGON ((6173564.792 8184143.987, 6173523.818...  

Células por UC (top 10):
 uc_id
0000.00.3633    18016
0000.00.3643    17255
0000.00.3642     3098
0000.00.3644     2194
0000.00.0187     1737
0000.00.0047     1518
0000.00.0173     1074
0000.00.0177     1032
0000.00.0114     1032
0000.00.0268      999
Name: count, dtype: int64


In [111]:
# Atribuir AAF às células com recorte real (overlay), evitando duplicação de área

gdf_aaf_clipped = gpd.overlay(
    gdf_aaf_metric,
    gdf_grid[['cell_id', 'uc_id', 'geometry']],
    how='intersection'
)

gdf_aaf_clipped['area_ha_cell'] = gdf_aaf_clipped.geometry.area / 10000

print("Total de eventos AAF originais:", len(gdf_aaf_metric))
print("Total de linhas após overlay (evento recortado por célula):", len(gdf_aaf_clipped))
print("\nExemplo:\n", gdf_aaf_clipped[['nome_uc', 'ano', 'area_ha', 'area_ha_cell', 'cell_id']].head())

Total de eventos AAF originais: 3133
Total de linhas após overlay (evento recortado por célula): 5652

Exemplo:
                          nome_uc   ano      area_ha  area_ha_cell  \
0  ESEC Serra Geral do Tocantins  2025  1134.594285     31.711126   
1  ESEC Serra Geral do Tocantins  2025  1134.594285    152.455354   
2  ESEC Serra Geral do Tocantins  2025  1134.594285     76.579652   
3  ESEC Serra Geral do Tocantins  2025  1134.594285    383.217311   
4  ESEC Serra Geral do Tocantins  2025  1134.594285    494.734471   

             cell_id  
0   0000.00.0076_3_9  
1  0000.00.0076_3_10  
2  0000.00.0076_3_11  
3   0000.00.0076_4_9  
4  0000.00.0076_4_10  


In [112]:
# Identificar eventos AAF que não caíram em nenhuma célula (fora da malha gerada)

events_with_match = gdf_aaf_clipped[['nome_uc', 'ano', 'area_ha']].drop_duplicates()
original_events = gdf_aaf_metric[['nome_uc', 'ano', 'area_ha']].drop_duplicates()

print("Eventos originais únicos:", len(original_events))
print("Eventos únicos com pelo menos 1 célula:", len(events_with_match))
print("Eventos sem nenhuma célula correspondente:", len(original_events) - len(events_with_match))

Eventos originais únicos: 3081
Eventos únicos com pelo menos 1 célula: 2909
Eventos sem nenhuma célula correspondente: 172


In [113]:
# Atribuir foco de calor INPE às células (ponto — sem risco de duplicação de área)

gdf_inpe_points = gpd.GeoDataFrame(
    df_inpe,
    geometry=gpd.points_from_xy(df_inpe['longitude'], df_inpe['latitude']),
    crs="EPSG:4326"
).to_crs(METRIC_CRS)

gdf_inpe_with_cell = gpd.sjoin(
    gdf_inpe_points,
    gdf_grid[['cell_id', 'uc_id', 'geometry']],
    how='inner',
    predicate='within'
)

print("Total de focos INPE (Brasil todo):", len(gdf_inpe_points))
print("Total de focos INPE dentro de alguma UC:", len(gdf_inpe_with_cell))
print("\nExemplo:\n", gdf_inpe_with_cell[['data_pas', 'frp', 'cell_id']].head())

Total de focos INPE (Brasil todo): 3466399
Total de focos INPE dentro de alguma UC: 131419

Exemplo:
                 data_pas  frp             cell_id
73   2025-01-01 03:23:00  3.5    0000.00.0007_3_1
76   2025-01-01 03:23:00  3.2    0000.00.0007_3_1
81   2025-01-01 03:23:00  3.2    0000.00.0007_3_1
105  2025-01-01 03:23:00  6.0    0000.00.0007_3_1
137  2025-01-01 03:46:00  4.7  0000.00.0009_12_23


In [ ]:
# Checar duplicidade de detecção simultânea por múltiplos satélites (mesmo ponto/instante)

inpe_duplicates = gdf_inpe_with_cell.duplicated(
    subset=['latitude', 'longitude', 'data_pas'], keep=False
)

print("Registros com mesma lat/lon/data_pas (possível duplicidade por satélite):", inpe_duplicates.sum())
print("\nExemplo:\n")
print(
    gdf_inpe_with_cell[inpe_duplicates]
    .sort_values(['latitude', 'longitude', 'data_pas'])
    [['latitude', 'longitude', 'data_pas', 'satelite', 'frp', 'cell_id']]
    .head(10)
)

Registros com mesma lat/lon/data_pas (possível duplicidade por satélite): 1298

Exemplo:

        latitude  longitude             data_pas satelite   frp  \
548303 -25.22405  -48.59155  2025-07-15 17:05:00  NOAA-20   0.2   
548303 -25.22405  -48.59155  2025-07-15 17:05:00  NOAA-20   0.2   
62307  -24.04483  -54.19886  2025-01-31 16:34:00  NPP-375  12.4   
62307  -24.04483  -54.19886  2025-01-31 16:34:00  NPP-375  12.4   
98927  -24.04455  -54.20038  2025-02-25 17:06:00  NPP-375   8.2   
98927  -24.04455  -54.20038  2025-02-25 17:06:00  NPP-375   8.2   
78552  -24.04366  -54.18759  2025-02-13 17:06:00  NOAA-21  16.0   
78552  -24.04366  -54.18759  2025-02-13 17:06:00  NOAA-21  16.0   
81764  -23.94108  -54.07211  2025-02-15 17:15:00  NOAA-20   4.2   
81764  -23.94108  -54.07211  2025-02-15 17:15:00  NOAA-20   4.2   

                 cell_id  
548303  0000.00.0017_3_6  
548303  0000.00.2634_1_5  
62307   0000.00.0025_3_0  
62307   0000.00.0161_1_0  
98927   0000.00.0025_3_0  
98927   00

: 

# Análise exploratória de dados (EDA)

# Pré-processamento

# Modelagem / treinamento

# Avaliação do modelo

# Ajuste de hiperparâmetros

# Validação final